In [1]:
import torch
import torch.nn.functional as F
import pandas as pd
import random

In [2]:
def load_data(path='worldcities.csv'):
    df = pd.read_csv(path)
    words = df['city_ascii'].dropna().tolist()
    print(f"Loaded {len(words)} city names")

    special_token = '<s>'
    vocab = [special_token] + sorted(set(''.join(words)))
    stoi = {s: i for i, s in enumerate(vocab)}
    itos = {i: s for s, i in stoi.items()}
    return words, vocab, stoi, itos

words, vocab, stoi, itos = load_data()
vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")
words[:5]

Loaded 48057 city names
Vocab size: 64


['Tokyo', 'Jakarta', 'Delhi', 'Guangzhou', 'Mumbai']

In [28]:
def build_dataset(words, block_size):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in list(w) + ['<s>']:
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X, Y = torch.tensor(X), torch.tensor(Y)
    return X, Y

def split_data(words, block_size, splits=(0.8, 0.9)):
    random.seed(42)
    random.shuffle(words)
    X, Y = build_dataset(words, block_size)
    n1 = int(splits[0] * X.shape[0])
    n2 = int(splits[1] * X.shape[0])
    Xtr, Xdev, Xte = X.tensor_split([n1, n2])
    Ytr, Ydev, Yte = Y.tensor_split([n1, n2])
    print(f"Xtr: {Xtr.shape}, Ytr: {Ytr.shape}")
    print(f"Xdev: {Xdev.shape}, Ydev: {Ydev.shape}")
    print(f"Xte: {Xte.shape}, Yte: {Yte.shape}")
    return (Xtr, Ytr), (Xdev, Ydev), (Xte, Yte)

In [4]:
def init_model(cfg):
    g = torch.Generator().manual_seed(cfg['seed'])
    bs, ed, hs = cfg['block_size'], cfg['emb_dim'], cfg['hidden_size']
    C  = torch.randn((vocab_size, ed), generator=g)
    W1 = torch.randn((ed * bs, hs), generator=g)
    b1 = torch.randn(hs, generator=g)
    W2 = torch.randn((hs, vocab_size), generator=g)
    b2 = torch.randn(vocab_size, generator=g)
    params = [C, W1, b1, W2, b2]
    for p in params:
        p.requires_grad = True
    print(f"Parameters: {sum(p.nelement() for p in params)}")
    return params

def forward(X, C, W1, b1, W2, b2):
    emb = C[X]
    h = torch.tanh(emb.view(-1, W1.shape[0]) @ W1 + b1)
    logits = h @ W2 + b2
    return logits

def train(cfg, params, train_set):
    C, W1, b1, W2, b2 = params
    Xtr, Ytr = train_set
    for i in range(cfg['train_steps']):
        ix = torch.randint(0, Xtr.shape[0], (cfg['batch_size'],))
        logits = forward(Xtr[ix], C, W1, b1, W2, b2)
        loss = F.cross_entropy(logits, Ytr[ix])
        for p in params:
            p.grad = None
        loss.backward()
        lr = cfg['lr_high'] if i < cfg['lr_switch'] else cfg['lr_low']
        for p in params:
            p.data += -lr * p.grad
        if (i + 1) % 20_000 == 0:
            print(f"Step {i+1:>7d}/{cfg['train_steps']}  loss: {loss.item():.4f}")

def evaluate(params, datasets):
    C, W1, b1, W2, b2 = params
    for name, (X, Y) in datasets.items():
        logits = forward(X, C, W1, b1, W2, b2)
        loss = F.cross_entropy(logits, Y)
        print(f"{name:10s} loss: {loss.item():.4f}")

def sample(cfg, params, count=None):
    C, W1, b1, W2, b2 = params
    g = torch.Generator().manual_seed(cfg['seed'] + 10)
    for _ in range(count or cfg['sample_count']):
        out, context = [], [0] * cfg['block_size']
        while True:
            emb = C[torch.tensor([context])]
            h = torch.tanh(emb.view(1, -1) @ W1 + b1)
            logits = h @ W2 + b2
            probs = F.softmax(logits, dim=1)
            ix = torch.multinomial(probs, num_samples=1, generator=g).item()
            context = context[1:] + [ix]
            if ix == 0:
                break
            out.append(ix)
        print(''.join(itos[i] for i in out))

In [27]:
# --- CONFIG (tweak and re-run from here) ---
CONFIG = dict(
    block_size=5,           # context length
    emb_dim=20,             # embedding dimensions
    hidden_size=300,        # hidden layer neurons
    train_steps=200_000,    # training iterations
    batch_size=64,
    lr_high=0.1,            # learning rate for first half
    lr_low=0.01,            # learning rate for second half
    lr_switch=100_000,       # step to switch lr
    seed=2147483647,
    sample_count=20,
)
train_set, dev_set, test_set = split_data(words, CONFIG['block_size'])
params = init_model(CONFIG)

X shape: torch.Size([484606, 5]), Y shape: torch.Size([484606])
Xtr: torch.Size([387684, 5]), Ytr: torch.Size([387684])
Xdev: torch.Size([48461, 5]), Ydev: torch.Size([48461])
Xte: torch.Size([48461, 5]), Yte: torch.Size([48461])
Parameters: 50844


In [22]:
train(CONFIG, params, train_set)

Step   20000/200000  loss: 2.8305
Step   40000/200000  loss: 2.7602
Step   60000/200000  loss: 2.7926
Step   80000/200000  loss: 2.8106
Step  100000/200000  loss: 2.7277
Step  120000/200000  loss: 2.4436
Step  140000/200000  loss: 2.5930
Step  160000/200000  loss: 2.3330
Step  180000/200000  loss: 2.5291
Step  200000/200000  loss: 2.3140


In [23]:
evaluate(params, {"train": train_set, "val": dev_set})

train      loss: 2.4643
val        loss: 2.5157


In [24]:
evaluate(params, {"test": test_set})

test       loss: 2.5221


In [25]:
sample(CONFIG, params)

Dhuglenn
Er Porto Elalol
Artee
Basar
Lasne
Saplelhuque
Pashal
Kralul
Raa E.genlyo Maceher
Morwill
Banchastan
Lugtia
Al en de Soiao
Lonchel
Sikupur
Hurt Teat
Gortyradu
Anefaing
Plarolanja
Gloleio
